# TGR-21 Day 3 · bean segmenter (masks + counting)
**Runtime → Change runtime type → A100 GPU** (L4 also works, slower). Upload `tgr_day3_colab.zip` to **My Drive** (top level) first, then **Runtime → Run all**.

Fine-tunes our Day 2 model on tight mixed piles built from the **real Day 3 beans with their hand-drawn outlines**, plus the real photos, directly at **896** (the size we run it at). The triangle photos are held out for validation.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/w && mkdir -p /content/w && cd /content/w && unzip -q /content/drive/MyDrive/tgr_day3_colab.zip
%cd /content/w
!mkdir -p /content/drive/MyDrive/tgr_day3
!ls
!pip -q install ultralytics

## 1. Build the dataset (~5–8 min)

In [ ]:
!python gen_dataset_d3.py --day1 day_1 --day3 day3 --labels labels.json --out beans_d3 --d3-piles 900 --mixed 200 --d1-piles 10 --val-piles 60 --real-repeat 4 --big-frac 0.25

## 2. Fine-tune at 896 (~45–75 min on A100)

In [ ]:
from ultralytics import YOLO
model = YOLO('best.pt')          # our Day 2 model (trained at 640)
model.train(data='beans_d3/data.yaml', imgsz=896, epochs=60, batch=16, workers=8, cache='ram',
            degrees=180, flipud=0.5, fliplr=0.5, hsv_h=0.01, hsv_s=0.3, hsv_v=0.3, mosaic=1.0, close_mosaic=10,
            patience=25, project='/content/drive/MyDrive/tgr_day3', name='seg896', exist_ok=True)

## 3. Validate + export for the Pi (NCNN) and laptops (ONNX)

In [ ]:
best = '/content/drive/MyDrive/tgr_day3/seg896/weights/best.pt'
m = YOLO(best)
met = m.val(data='beans_d3/data.yaml', imgsz=896)
print('mask mAP50', met.seg.map50, ' mask mAP50-95', met.seg.map)
m.export(format='onnx', imgsz=896, simplify=True)
m.export(format='ncnn', imgsz=896)
!cd /content/drive/MyDrive/tgr_day3/seg896/weights && zip -qr /content/drive/MyDrive/tgr_day3/day3_seg_weights.zip best.pt best.onnx best_ncnn_model
!ls -la /content/drive/MyDrive/tgr_day3/

Download **`tgr_day3/day3_seg_weights.zip`** from My Drive into `Downloads/Day3` on the Mac and tell Claude.